# Analisis Anggaran dan Realisasi APBD Indonesia

Notebook ini mengeksplorasi anggaran dan realisasi APBD pemerintah daerah di Indonesia menggunakan data yang telah dibersihkan. Analisis mencakup tren bulanan dan tahunan, perbandingan antarprovinsi, serta perbandingan berdasarkan jenis akun.

Persentase realisasi dihitung sebagai **total realisasi dibagi total anggaran, kemudian dikalikan 100%**. Tahun 2026 belum mencakup satu tahun anggaran penuh sehingga hasilnya perlu ditafsirkan secara terpisah dan tidak dibandingkan langsung dengan tahun penuh.

## 1. Tren Bulanan Anggaran dan Realisasi APBD (2023?2026)

Visualisasi berikut membandingkan anggaran dan realisasi bulanan pada periode yang tersedia.

In [3]:
from pathlib import Path
import polars as pl
import duckdb
import plotly.express as px

DATA_PATH = Path(
    r"D:\SEMESTER 7\Analisis Big Data\tugas1-abd-2026\data\processed\apbd_pemda_indonesia_clean.parquet"
)

# Pastikan file ditemukan
print("File ditemukan:", DATA_PATH.exists())

# Baca data secara lazy
df = pl.scan_parquet(DATA_PATH)

# Tampilkan ringkasan data
display(
    df.select([
        pl.len().alias("jumlah_baris"),
        pl.col("tahun").min().alias("tahun_awal"),
        pl.col("tahun").max().alias("tahun_akhir"),
        pl.col("tahun").n_unique().alias("jumlah_tahun"),
        pl.col("provinsi").n_unique().alias("jumlah_provinsi"),
        pl.col("pemda").n_unique().alias("jumlah_pemda"),
    ]).collect()
)

# Tampilkan struktur kolom
print(df.collect_schema())

File ditemukan: True


jumlah_baris,tahun_awal,tahun_akhir,jumlah_tahun,jumlah_provinsi,jumlah_pemda
u32,i64,i64,u32,u32,u32
905781,2011,2026,16,38,587


Schema({'tahun': Int64, 'bulan': Int64, 'id_provinsi': Int64, 'provinsi': String, 'id_pemda': Int64, 'pemda': String, 'no_akun': Int64, 'akun': String, 'anggaran': Float64, 'realisasi': Float64, 'persentase': Float64})


In [4]:
# Agregasi berdasarkan tahun dan bulan
df_bulanan = (
    df
    .group_by(["tahun", "bulan"])
    .agg([
        pl.col("anggaran").sum().alias("total_anggaran"),
        pl.col("realisasi").sum().alias("total_realisasi"),
    ])
    .with_columns([
        (
            pl.col("total_realisasi")
            / pl.col("total_anggaran")
            * 100
        ).alias("persentase_realisasi")
    ])
    .sort(["tahun", "bulan"])
    .collect()
)

# Tampilkan hasil
display(df_bulanan.head(15))

print("Jumlah periode bulanan:", df_bulanan.height)

tahun,bulan,total_anggaran,total_realisasi,persentase_realisasi
i64,i64,f64,f64,f64
2011,12,3.0038e15,3.2554e15,108.374363
2012,12,3.6131e15,3.9058e15,108.101535
2013,12,4.3158e15,4.4708e15,103.593198
2014,12,5.0135e15,5.0241e15,100.211502
2015,12,5.5863e15,5.6165e15,100.540851
…,…,…,…,…
2021,12,7.2526e15,7.0742e15,97.540705
2022,12,7.0512e15,7.3123e15,103.703465
2023,1,7.8146e15,4.4833e14,5.737003


Jumlah periode bulanan: 56


In [ ]:
# Fokus pada data tahun 2023 ke atas
df_bulanan_modern = (
    df_bulanan
    .filter(pl.col("tahun") >= 2023)
    .with_columns(
        pl.date(
            pl.col("tahun"),
            pl.col("bulan"),
            pl.lit(1)
        ).alias("tanggal")
    )
    .sort("tanggal")
)

# Visualisasi tren anggaran
fig = px.line(
    df_bulanan_modern,
    x="tanggal",
    y="total_anggaran",
    title="Tren Anggaran APBD Indonesia (2023–2026)",
    markers=True,
    labels={
        "tanggal": "Periode",
        "total_anggaran": "Total Anggaran (Rupiah)"
    }
)

fig.update_layout(
    width=1000,
    height=500,
    hovermode="x unified"
)

fig.show()


# Visualisasi tren realisasi
fig = px.line(
    df_bulanan_modern,
    x="bulan",
    y="total_realisasi",
    color="tahun",
    title="Tren Realisasi APBD Indonesia per Tahun (2023–2026)",
    markers=True,
    labels={
        "bulan": "Bulan",
        "total_realisasi": "Total Realisasi (Rupiah)",
        "tahun": "Tahun"
    }
)

fig.update_layout(
    width=1000,
    height=500,
    hovermode="x unified",
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)

fig.show()


### Interpretasi Tren Bulanan

Grafik memperlihatkan perubahan anggaran dan realisasi APBD dari bulan ke bulan pada periode 2023?2026. Beberapa hal yang perlu diperhatikan:

- Nilai anggaran berubah antarbulan dan antartahun; pola anggaran tidak selalu sama dengan pola realisasi.
- Nilai realisasi yang terakumulasi cenderung bertambah sepanjang tahun. Kenaikan ini perlu dibaca sebagai akumulasi, bukan sebagai ukuran laju penyerapan bulanan.
- Kelengkapan periode berbeda: data 2023 tersedia hingga November, data 2024 dan 2025 mencakup 12 bulan, sedangkan data 2026 tersedia hingga September.

Karena panjang periode pengamatan tidak sama, total antar-tahun tidak dapat dibandingkan secara langsung tanpa mempertimbangkan bulan yang tercakup.

## 2. Perbandingan Anggaran dan Realisasi APBD Antarprovinsi

Bagian ini merangkum total anggaran, total realisasi, dan persentase realisasi tiap provinsi. Nilai total menunjukkan besaran nominal, sedangkan persentase menunjukkan perbandingan realisasi terhadap anggaran.

In [11]:
# Agregasi anggaran dan realisasi berdasarkan provinsi
df_provinsi = (
    df
    .group_by("provinsi")
    .agg([
        pl.col("anggaran").sum().alias("total_anggaran"),
        pl.col("realisasi").sum().alias("total_realisasi"),
    ])
    .with_columns(
        (
            pl.col("total_realisasi") /
            pl.col("total_anggaran") * 100
        ).alias("persentase_realisasi")
    )
    .sort("total_anggaran", descending=True)
)

# Eksekusi query dan tampilkan hasil
df_provinsi = df_provinsi.collect()

display(df_provinsi)

provinsi,total_anggaran,total_realisasi,persentase_realisasi
str,f64,f64,f64
"""Provinsi Jawa Timur""",4.5690e16,2.4848e16,54.384795
"""Provinsi Jawa Barat""",4.4811e16,2.4347e16,54.331896
"""Provinsi Jawa Tengah""",3.6846e16,2.0293e16,55.075242
"""Provinsi DKI Jakarta""",2.6325e16,1.2025e16,45.679256
"""Provinsi Kalimantan Timur""",2.0571e16,9.9979e15,48.6028
…,…,…,…
"""Provinsi Bangka Belitung""",3.2068e15,1.7994e15,56.111562
"""Provinsi Sulawesi Barat""",2.6599e15,1.3876e15,52.169233
"""Provinsi Gorontalo""",2.5444e15,1.4197e15,55.79619


In [12]:
# Ambil 10 provinsi dengan total anggaran terbesar
df_top10 = df_provinsi.head(10)

# Visualisasi perbandingan anggaran dan realisasi
fig = px.bar(
    df_top10,
    x="total_anggaran",
    y="provinsi",
    orientation="h",
    barmode="group",
    title="Top 10 Provinsi Berdasarkan Total Anggaran APBD",
    labels={
        "total_anggaran": "Total Anggaran (Rupiah)",
        "provinsi": "Provinsi"
    }
)

fig.update_layout(
    width=1000,
    height=600,
    yaxis={"categoryorder": "total ascending"},
    hovermode="y unified"
)

fig.show()

In [13]:
# Urutkan provinsi berdasarkan persentase realisasi
df_realisasi_provinsi = (
    df_provinsi
    .sort("persentase_realisasi", descending=True)
)

# Visualisasi persentase realisasi
fig = px.bar(
    df_realisasi_provinsi,
    x="persentase_realisasi",
    y="provinsi",
    orientation="h",
    title="Persentase Realisasi APBD Berdasarkan Provinsi",
    labels={
        "persentase_realisasi": "Persentase Realisasi (%)",
        "provinsi": "Provinsi"
    },
)

fig.update_layout(
    width=1100,
    height=900,
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Persentase Realisasi (%)",
    hovermode="y unified"
)

fig.show()

### Interpretasi Hasil Antarprovinsi

Pada agregasi data yang ditampilkan, Kalimantan Tengah memiliki persentase realisasi tertinggi, sedangkan Papua Pegunungan memiliki persentase terendah. Jawa Timur, Jawa Barat, dan Jawa Tengah menunjukkan persentase yang relatif berdekatan.

Perbandingan total anggaran atau realisasi berbeda maknanya dari perbandingan persentase realisasi. Besarnya total tidak selalu diikuti oleh persentase penyerapan yang tinggi. Hasil agregasi lintas tahun juga perlu ditafsirkan dengan hati-hati karena periode dan kelengkapan data dapat berbeda antarwilayah.

In [14]:
# Mengurutkan provinsi berdasarkan total realisasi
top10_realisasi = (
    df_provinsi
    .sort("total_realisasi", descending=True)
    .head(10)
)

display(top10_realisasi)

# Visualisasi Top 10 provinsi
fig = px.bar(
    top10_realisasi,
    x="total_realisasi",
    y="provinsi",
    orientation="h",
    title="Top 10 Provinsi Berdasarkan Total Realisasi APBD",
    labels={
        "provinsi": "Provinsi",
        "total_realisasi": "Total Realisasi (Rupiah)"
    }
)

fig.update_layout(
    width=1000,
    height=550,
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

provinsi,total_anggaran,total_realisasi,persentase_realisasi
str,f64,f64,f64
"""Provinsi Jawa Timur""",4.5690e16,2.4848e16,54.384795
"""Provinsi Jawa Barat""",4.4811e16,2.4347e16,54.331896
"""Provinsi Jawa Tengah""",3.6846e16,2.0293e16,55.075242
"""Provinsi DKI Jakarta""",2.6325e16,1.2025e16,45.679256
"""Provinsi Sumatera Utara""",2.0496e16,1.0638e16,51.905693
"""Provinsi Kalimantan Timur""",2.0571e16,9.9979e15,48.6028
"""Provinsi Sulawesi Selatan""",1.5027e16,7.7337e15,51.466292
"""Provinsi Sumatera Selatan""",1.4633e16,7.5973e15,51.918743
"""Provinsi Aceh""",1.3663e16,7.3080e15,53.485967


## 3. Anggaran dan Realisasi APBD Berdasarkan Jenis Akun

Bagian ini membandingkan total anggaran dan total realisasi untuk setiap jenis akun selama periode pengamatan. Perbandingan nilai total menunjukkan besaran nominal, bukan tingkat penyerapan anggaran.

In [15]:
# Agregasi anggaran dan realisasi berdasarkan jenis akun
df_akun = (
    df.group_by("akun")
    .agg([
        pl.col("anggaran").sum().alias("total_anggaran"),
        pl.col("realisasi").sum().alias("total_realisasi")
    ])
    .sort("total_realisasi", descending=True)
    .collect()
)

display(df_akun.head(10))

akun,total_anggaran,total_realisasi
str,f64,f64
"""Pendapatan Daerah""",6.4452e16,3.4386e16
"""Belanja Daerah""",6.7765e16,2.9917e16
"""Belanja Pegawai""",4.7684e16,2.3422e16
"""TKDD""",4.1326e16,2.2574e16
"""Pendapatan Transfer Pemerintah…",4.1326e16,2.2574e16
"""Belanja Barang dan Jasa""",3.6858e16,1.4245e16
"""PAD""",2.0150e16,1.0776e16
"""Belanja Modal""",2.2042e16,7.9133e15
"""Pajak Daerah""",1.4412e16,7.7141e15


In [16]:
# Mengambil 10 akun dengan total realisasi terbesar
top10_akun = df_akun.head(10).sort(
    "total_realisasi"
)

fig = px.bar(
    top10_akun,
    x="total_realisasi",
    y="akun",
    orientation="h",
    title="Top 10 Jenis Akun Berdasarkan Total Realisasi APBD",
    labels={
        "akun": "Jenis Akun",
        "total_realisasi": "Total Realisasi (Rupiah)"
    }
)

fig.update_layout(
    width=1100,
    height=600,
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

### Persentase Realisasi Berdasarkan Jenis Akun

Persentase realisasi setiap akun dihitung dari total realisasi dibagi total anggaran pada akun tersebut, kemudian dikalikan 100%. Grafik menampilkan akun dengan persentase tertinggi; nilai persentase perlu dibaca bersama nilai anggaran dan realisasinya agar besaran nominalnya tetap terlihat.

In [17]:
# Menghitung persentase realisasi berdasarkan jenis akun
df_persentase_akun = (
    df.group_by("akun")
    .agg([
        pl.col("anggaran").sum().alias("total_anggaran"),
        pl.col("realisasi").sum().alias("total_realisasi")
    ])
    .with_columns(
        pl.when(pl.col("total_anggaran") != 0)
        .then(
            pl.col("total_realisasi")
            / pl.col("total_anggaran") * 100
        )
        .otherwise(None)
        .alias("persentase_realisasi")
    )
    .sort("persentase_realisasi", descending=True)
    .collect()
)

display(df_persentase_akun.head(10))

akun,total_anggaran,total_realisasi,persentase_realisasi
str,f64,f64,f64
"""Penerimaan Piutang Daerah""",0.0,6.1550e9,null
"""Dana Darurat""",1.9381e12,4.1006e12,211.577732
"""SiLPA TA Sebelumnya""",2.8789e14,4.4547e14,154.737224
"""Penerimaan Pembiayaan""",3.1163e14,4.5897e14,147.281299
"""Pembiayaan""",3.4263e14,4.7550e14,138.780062
"""Penjualan Kekayaan Daerah yang…",9.1431e11,1.2075e12,132.062295
"""Bagi Hasil Provinsi atau Pemda…",1.0716e14,1.3255e14,123.697847
"""Dana Penyesuaian dan Otsus""",3.6006e14,4.3555e14,120.965905
"""DAK""",1.3839e14,1.5448e14,111.621375


In [20]:
# Mengambil 10 akun dengan persentase realisasi tertinggi
top10_persentase = (
    df_persentase_akun
    .filter(pl.col("persentase_realisasi").is_not_null())
    .sort("persentase_realisasi", descending=True)
    .head(10)
    .sort("persentase_realisasi")
)

# Visualisasi
fig = px.bar(
    top10_persentase,
    x="persentase_realisasi",
    y="akun",
    orientation="h",
    title="Top 10 Jenis Akun Berdasarkan Persentase Realisasi APBD",
    labels={
        "akun": "Jenis Akun",
        "persentase_realisasi": "Persentase Realisasi (%)"
    },
    text="persentase_realisasi"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    width=1100,
    height=600,
    xaxis_title="Persentase Realisasi (%)",
    yaxis_title="Jenis Akun"
)

fig.update_xaxes(
    range=[
        0,
        top10_persentase["persentase_realisasi"].max() * 1.15
    ]
)

fig.show()

## 4. Persentase Realisasi APBD Antarprovinsi

Bagian ini membandingkan rasio total realisasi terhadap total anggaran untuk tiap provinsi. Selain melihat provinsi dengan persentase tertinggi, analisis juga menyoroti provinsi dengan persentase terendah dan rentang di antara keduanya.

In [21]:
df_persentase_provinsi = (
    df.group_by("provinsi")
    .agg([
        pl.col("anggaran").sum().alias("total_anggaran"),
        pl.col("realisasi").sum().alias("total_realisasi")
    ])
    .with_columns(
        pl.when(pl.col("total_anggaran") != 0)
        .then(
            pl.col("total_realisasi")
            / pl.col("total_anggaran") * 100
        )
        .otherwise(None)
        .alias("persentase_realisasi")
    )
    .sort("persentase_realisasi", descending=True)
    .collect()
)

display(df_persentase_provinsi)

provinsi,total_anggaran,total_realisasi,persentase_realisasi
str,f64,f64,f64
"""Provinsi Kalimantan Tengah""",9.7905e15,7.0088e15,71.58808
"""Provinsi Papua""",4.9896e15,2.9062e15,58.245597
"""Provinsi Bangka Belitung""",3.2068e15,1.7994e15,56.111562
"""Provinsi DI Yogyakarta""",5.5832e15,3.1184e15,55.852479
"""Provinsi Gorontalo""",2.5444e15,1.4197e15,55.79619
…,…,…,…
"""Provinsi Maluku Utara""",5.3059e15,2.3059e15,43.458785
"""Provinsi Papua Barat Daya""",2.4029e15,8.7878e14,36.571912
"""Provinsi Papua Selatan""",2.2115e15,8.0050e14,36.197719


In [22]:
# Mengambil 10 provinsi dengan persentase realisasi tertinggi
top10_persentase_provinsi = (
    df_persentase_provinsi
    .filter(pl.col("persentase_realisasi").is_not_null())
    .sort("persentase_realisasi", descending=True)
    .head(10)
    .sort("persentase_realisasi")
)

# Visualisasi
fig = px.bar(
    top10_persentase_provinsi,
    x="persentase_realisasi",
    y="provinsi",
    orientation="h",
    title="Top 10 Provinsi Berdasarkan Persentase Realisasi APBD",
    labels={
        "provinsi": "Provinsi",
        "persentase_realisasi": "Persentase Realisasi (%)"
    },
    text="persentase_realisasi"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    width=1100,
    height=600,
    xaxis_title="Persentase Realisasi (%)",
    yaxis_title="Provinsi"
)

fig.update_xaxes(
    range=[
        0,
        top10_persentase_provinsi["persentase_realisasi"].max() * 1.15
    ]
)

fig.show()

### 10 Provinsi dengan Persentase Realisasi Terendah

Bagian ini menampilkan sepuluh provinsi dengan persentase realisasi terendah berdasarkan rasio total realisasi terhadap total anggaran. Urutan tersebut menunjukkan perbandingan rasio dalam data yang dianalisis, bukan penilaian atas penyebab atau kualitas pengelolaan anggaran.

In [23]:
# Mengambil 10 provinsi dengan persentase realisasi terendah
bottom10_provinsi = (
    df_persentase_provinsi
    .filter(pl.col("persentase_realisasi").is_not_null())
    .sort("persentase_realisasi")
    .head(10)
)

display(bottom10_provinsi)

provinsi,total_anggaran,total_realisasi,persentase_realisasi
str,f64,f64,f64
"""Provinsi Papua Pegunungan""",3.6374e15,1.1578e15,31.830805
"""Provinsi Papua Tengah""",5.3609e15,1.9027e15,35.490957
"""Provinsi Papua Selatan""",2.2115e15,8.0050e14,36.197719
"""Provinsi Papua Barat Daya""",2.4029e15,8.7878e14,36.571912
"""Provinsi Maluku Utara""",5.3059e15,2.3059e15,43.458785
"""Provinsi DKI Jakarta""",2.6325e16,1.2025e16,45.679256
"""Provinsi Maluku""",4.6854e15,2.2639e15,48.317343
"""Provinsi Kalimantan Timur""",2.0571e16,9.9979e15,48.6028
"""Provinsi Papua Barat""",4.8588e15,2.3896e15,49.181709


In [24]:
fig = px.bar(
    bottom10_provinsi.sort("persentase_realisasi", descending=True),
    x="persentase_realisasi",
    y="provinsi",
    orientation="h",
    title="10 Provinsi dengan Persentase Realisasi APBD Terendah",
    labels={
        "provinsi": "Provinsi",
        "persentase_realisasi": "Persentase Realisasi (%)"
    },
    text="persentase_realisasi"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    width=1100,
    height=600,
    xaxis_title="Persentase Realisasi (%)",
    yaxis_title="Provinsi"
)

fig.show()

### Rentang Persentase Realisasi Antarprovinsi

Perbandingan nilai tertinggi dan terendah digunakan untuk menggambarkan rentang persentase realisasi antarprovinsi. Selisih dinyatakan dalam **poin persentase**, bukan persen relatif.

In [25]:
persentase_tertinggi = (
    df_persentase_provinsi
    .select(pl.col("persentase_realisasi").max())
    .item()
)

persentase_terendah = (
    df_persentase_provinsi
    .select(pl.col("persentase_realisasi").min())
    .item()
)

print(f"Persentase tertinggi : {persentase_tertinggi:.2f}%")
print(f"Persentase terendah  : {persentase_terendah:.2f}%")
print(
    f"Kesenjangan realisasi: "
    f"{persentase_tertinggi - persentase_terendah:.2f} poin persentase"
)

Persentase tertinggi : 71.59%
Persentase terendah  : 31.83%
Kesenjangan realisasi: 39.76 poin persentase


### Ringkasan Hasil Antarprovinsi

Berdasarkan agregasi data periode 2011?2026, Kalimantan Tengah memiliki persentase realisasi tertinggi sebesar **71,59%**, sedangkan Papua Pegunungan memiliki persentase terendah sebesar **31,83%**.

Selisih keduanya adalah **39,76 poin persentase**. Angka ini merangkum rentang yang terlihat pada data, tetapi tidak menjelaskan faktor penyebab perbedaan. Perbandingan perlu mempertimbangkan perbedaan periode dan kelengkapan data.

## 5. Tren Tahunan Anggaran dan Realisasi APBD

Bagian ini merangkum total anggaran dan realisasi APBD per tahun untuk melihat perubahan dari waktu ke waktu. Tahun 2026 tidak dimasukkan dalam perbandingan tahunan penuh karena datanya belum mencakup seluruh tahun anggaran.

In [27]:
df_tahunan = (
    df_bulanan
    .group_by("tahun")
    .agg([
        pl.col("total_anggaran").max().alias("total_anggaran"),
        pl.col("total_realisasi").max().alias("total_realisasi")
    ])
    .sort("tahun")
)

display(df_tahunan)

tahun,total_anggaran,total_realisasi
i64,f64,f64
2011,3.0038e15,3.2554e15
2012,3.6131e15,3.9058e15
2013,4.3158e15,4.4708e15
2014,5.0135e15,5.0241e15
2015,5.5863e15,5.6165e15
…,…,…
2022,7.0512e15,7.3123e15
2023,7.8146e15,7.5189e15
2024,8.7105e15,8.4720e15


In [28]:
df_tahunan = (
    df_tahunan
    .with_columns(
        pl.when(pl.col("total_anggaran") != 0)
        .then(
            pl.col("total_realisasi")
            / pl.col("total_anggaran") * 100
        )
        .otherwise(None)
        .alias("persentase_realisasi")
    )
)

display(df_tahunan)

tahun,total_anggaran,total_realisasi,persentase_realisasi
i64,f64,f64,f64
2011,3.0038e15,3.2554e15,108.374363
2012,3.6131e15,3.9058e15,108.101535
2013,4.3158e15,4.4708e15,103.593198
2014,5.0135e15,5.0241e15,100.211502
2015,5.5863e15,5.6165e15,100.540851
…,…,…,…
2022,7.0512e15,7.3123e15,103.703465
2023,7.8146e15,7.5189e15,96.215245
2024,8.7105e15,8.4720e15,97.262749


In [30]:
# Grafik tren realisasi 2011–2025
df_tahunan_lengkap = df_tahunan.filter(
    pl.col("tahun") <= 2025
)

fig = px.line(
    df_tahunan_lengkap,
    x="tahun",
    y="total_realisasi",
    markers=True,
    title="Tren Realisasi APBD Indonesia (2011–2025)",
    labels={
        "tahun": "Tahun",
        "total_realisasi": "Total Realisasi (Rupiah)"
    }
)

fig.update_layout(
    width=1100,
    height=550,
    hovermode="x unified"
)

fig.show()

### Interpretasi Tren Tahunan

Pada periode 2011?2025, grafik menunjukkan kecenderungan kenaikan realisasi dalam jangka panjang dengan perubahan pada beberapa tahun:

- Realisasi meningkat secara bertahap pada sebagian besar periode 2011?2019.
- Pada 2020, realisasi menurun dibandingkan tahun sebelumnya.
- Sejak 2021, realisasi kembali menunjukkan kecenderungan meningkat.
- Pada grafik yang ditampilkan, total realisasi tertinggi terjadi pada 2025, sekitar **Rp10,27 kuadriliun**.

**Catatan:** Data 2026 tidak disertakan dalam perbandingan ini karena hanya mencakup Januari?September, sehingga belum sebanding dengan tahun anggaran penuh.

In [31]:
# Mengambil data tahun 2011–2025
df_perbandingan_tahunan = (
    df_tahunan
    .filter(pl.col("tahun") <= 2025)
)

# Visualisasi perbandingan anggaran dan realisasi
fig = px.line(
    df_perbandingan_tahunan,
    x="tahun",
    y=["total_anggaran", "total_realisasi"],
    markers=True,
    title="Perbandingan Anggaran dan Realisasi APBD Indonesia (2011–2025)",
    labels={
        "tahun": "Tahun",
        "value": "Jumlah (Rupiah)",
        "variable": "Keterangan"
    }
)

fig.update_layout(
    width=1100,
    height=550,
    hovermode="x unified",
    legend_title="Keterangan"
)

fig.show()

In [32]:
# Menghitung selisih anggaran dan realisasi tahunan
df_gap_tahunan = (
    df_tahunan
    .filter(pl.col("tahun") <= 2025)
    .with_columns([
        (
            pl.col("total_realisasi") -
            pl.col("total_anggaran")
        ).alias("selisih_realisasi"),

        (
            pl.col("total_realisasi") /
            pl.col("total_anggaran") * 100
        ).alias("persentase_realisasi")
    ])
    .sort("tahun")
)

display(df_gap_tahunan)

tahun,total_anggaran,total_realisasi,persentase_realisasi,selisih_realisasi
i64,f64,f64,f64,f64
2011,3.0038e15,3.2554e15,108.374363,2.5155e14
2012,3.6131e15,3.9058e15,108.101535,2.9272e14
2013,4.3158e15,4.4708e15,103.593198,1.5507e14
2014,5.0135e15,5.0241e15,100.211502,1.0604e13
2015,5.5863e15,5.6165e15,100.540851,3.0214e13
…,…,…,…,…
2021,7.2526e15,7.0742e15,97.540705,-1.7836e14
2022,7.0512e15,7.3123e15,103.703465,2.6114e14
2023,7.8146e15,7.5189e15,96.215245,-2.9576e14


In [33]:
# Mengubah selisih ke satuan kuadriliun rupiah
df_gap_visual = (
    df_gap_tahunan
    .with_columns(
        (
            pl.col("selisih_realisasi") / 1e15
        ).alias("selisih_kuadriliun")
    )
    .with_columns(
        pl.when(pl.col("selisih_kuadriliun") >= 0)
        .then(pl.lit("Realisasi di atas anggaran"))
        .otherwise(pl.lit("Realisasi di bawah anggaran"))
        .alias("kategori")
    )
)

fig = px.bar(
    df_gap_visual,
    x="tahun",
    y="selisih_kuadriliun",
    color="kategori",
    title="Kesenjangan Anggaran dan Realisasi APBD (2011–2025)",
    labels={
        "tahun": "Tahun",
        "selisih_kuadriliun": "Selisih (Rp Kuadriliun)",
        "kategori": "Keterangan"
    },
    barmode="relative"
)

fig.update_layout(
    width=1100,
    height=550,
    xaxis_title="Tahun",
    yaxis_title="Selisih (Rp Kuadriliun)",
    hovermode="x unified"
)

fig.add_hline(y=0, line_dash="dash", line_color="black")

fig.show()

### Kesenjangan antara Anggaran dan Realisasi

Grafik selisih anggaran dan realisasi menunjukkan bahwa hubungan keduanya berubah dari tahun ke tahun. Pada sejumlah tahun, realisasi berada di bawah anggaran, sedangkan pada tahun lainnya realisasi melampaui anggaran. Berdasarkan visualisasi, selisih negatif terbesar terjadi pada 2020 dan selisih positif terbesar pada 2025.

Perubahan selisih dapat menjadi bahan evaluasi perencanaan dan pelaksanaan APBD, tetapi tidak cukup untuk menyimpulkan penyebabnya.

**Catatan:** Selisih positif tidak otomatis menunjukkan kinerja yang lebih baik. Interpretasi perlu mempertimbangkan karakteristik akun serta konsistensi pencatatan anggaran dan realisasi.

## Kesimpulan

Analisis data APBD pemerintah daerah Indonesia periode 2011?2026 menghasilkan beberapa ringkasan berikut:

1. **Tren tahunan:** Realisasi secara umum menunjukkan kecenderungan meningkat, meskipun terdapat fluktuasi pada beberapa tahun.
2. **Anggaran dan realisasi:** Selisih antara keduanya berubah antarperiode; realisasi tidak selalu berada di bawah anggaran.
3. **Perbandingan antarprovinsi:** Dalam agregasi yang ditampilkan, Kalimantan Tengah memiliki persentase realisasi tertinggi (**71,59%**) dan Papua Pegunungan terendah (**31,83%**), dengan selisih **39,76 poin persentase**.
4. **Perbandingan jenis akun:** Pendapatan Daerah memiliki total realisasi terbesar di antara akun yang ditampilkan, sedangkan Dana Darurat memiliki persentase realisasi tertinggi di antara akun tersebut.

Secara keseluruhan, visualisasi menggambarkan perubahan realisasi dari waktu ke waktu serta perbedaan nilai dan persentase realisasi antarprovinsi dan jenis akun. Temuan ini bersifat deskriptif dan tidak dengan sendirinya menjelaskan faktor penyebabnya.

**Batasan:** Data 2026 belum mencakup satu tahun anggaran penuh. Karena itu, nilainya perlu ditafsirkan secara terpisah dari tahun-tahun penuh; agregasi lintas tahun juga dapat dipengaruhi oleh perbedaan cakupan data.